# 02_agrupar_por_lat_lon

Este notebook carga los datos limpios generados en `01_read_filter_data.ipynb`, crea una rejilla de latitud/longitud y resume las especies más abundantes por cuadro.

- `lat`, `long`: rejilla redondeada a 1 decimal.
- `especie`: especie dominante en el cuadro.
- `cantidad`: número de registros de esa especie en el cuadro.


In [1]:
import os

jdk_path = os.path.expanduser("~/.jdk17")
if os.path.isdir(jdk_path):
    os.environ["JAVA_HOME"] = jdk_path
    os.environ["PATH"] = os.path.join(jdk_path, "bin") + os.pathsep + os.environ.get("PATH", "")
    print("JAVA_HOME seteado a", jdk_path)
else:
    print("No se encontró JDK en", jdk_path)


JAVA_HOME seteado a /home/codespace/.jdk17


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, round, desc, row_number
from pyspark.sql.window import Window
import os

spark = SparkSession.builder \
    .appName("BiodiversidadColombiaGrid") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

clean_parquet = os.path.join(os.getcwd(), "biodiversidad_limpa.parquet")
print("Leyendo datos limpios desde:", clean_parquet)
df = spark.read.parquet(clean_parquet)

# Crear la rejilla espacial redondeando a 1 decimal
df_grid = df \
    .withColumn("lat", round(col("decimalLatitude"), 1)) \
    .withColumn("long", round(col("decimalLongitude"), 1))

# Contar especies por cuadro
conteo_por_cuadro = df_grid \
    .groupBy("lat", "long", "species") \
    .count() \
    .withColumnRenamed("count", "cantidad")

# Seleccionar la especie más abundante en cada cuadro
ventana = Window.partitionBy("lat", "long").orderBy(desc("cantidad"))
tabla_cuadros = conteo_por_cuadro \
    .withColumn("rank", row_number().over(ventana)) \
    .filter(col("rank") == 1) \
    .drop("rank") \
    .select(
        col("lat"),
        col("long"),
        col("species").alias("especie"),
        col("cantidad")
    ) \
    .orderBy(desc("cantidad"))

print("Top 50 cuadros con las especies más abundantes:")
tabla_cuadros.show(50, truncate=False)

salida_parquet = os.path.join(os.getcwd(), "datos_para_mapa_rejilla.parquet")
tabla_cuadros.write.mode("overwrite").parquet(salida_parquet)
print("Guardado el resumen de cuadriculado en:", salida_parquet)

spark.stop()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/29 05:55:56 WARN Utils: Your hostname, codespaces-e48955, resolves to a loopback address: 127.0.0.1; using 10.0.0.90 instead (on interface eth0)
26/05/29 05:55:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/29 05:55:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/29 05:55:59 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Leyendo datos limpios desde: /workspaces/bigData-Class/Notebook/biodiversidad_limpa.parquet


Top 50 cuadros con las especies más abundantes:


+----+-----+-------------------------+--------+
|lat |long |especie                  |cantidad|
+----+-----+-------------------------+--------+
|11.1|-72.7|Vachellia macracantha    |5316    |
|10.1|-73.9|Astronium graveolens     |1999    |
|11.0|-72.7|unknown                  |1519    |
|10.1|-73.8|Astronium graveolens     |1443    |
|10.8|-72.9|unknown                  |1382    |
|10.4|-73.2|unknown                  |1280    |
|10.3|-73.2|unknown                  |1225    |
|6.4 |-76.4|unknown                  |1103    |
|10.1|-73.7|unknown                  |1073    |
|10.2|-74.0|Guazuma ulmifolia        |1067    |
|10.7|-73.0|Copernicia tectorum      |973     |
|6.5 |-76.3|unknown                  |697     |
|10.1|-73.6|Handroanthus billbergii  |675     |
|10.2|-73.6|Guazuma ulmifolia        |599     |
|11.2|-72.6|Handroanthus billbergii  |571     |
|10.2|-73.4|Guazuma ulmifolia        |554     |
|1.2 |-78.0|unknown                  |548     |
|4.0 |-76.2|Chamaedorea tepejilote   |53

Guardado el resumen de cuadriculado en: /workspaces/bigData-Class/Notebook/datos_para_mapa_rejilla.parquet


## Ajustar el tamaño del cuadro

Si quieres cuadros más pequeños, cambia `round(..., 1)` a `round(..., 2)` en la celda anterior.

- `1 decimal` ≈ cuadros regionales (~10 km).
- `2 decimales` ≈ cuadros más finos (~1 km).


In [3]:
from pyspark.sql import SparkSession
import os

spark = SparkSession.builder \
    .appName("BiodiversidadColombiaGridViewer") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

parquet_path = os.path.join(os.getcwd(), "datos_para_mapa_rejilla.parquet")
print("Leyendo resumen de cuadriculado desde:", parquet_path)
df = spark.read.parquet(parquet_path)

print("Total cuadros:", df.count())
df.show(50, truncate=False)

spark.stop()


26/05/29 05:57:38 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Leyendo resumen de cuadriculado desde: /workspaces/bigData-Class/Notebook/datos_para_mapa_rejilla.parquet
Total cuadros: 2306
+----+-----+-------------------------+--------+
|lat |long |especie                  |cantidad|
+----+-----+-------------------------+--------+
|11.1|-72.7|Vachellia macracantha    |5316    |
|10.1|-73.9|Astronium graveolens     |1999    |
|11.0|-72.7|unknown                  |1519    |
|10.1|-73.8|Astronium graveolens     |1443    |
|10.8|-72.9|unknown                  |1382    |
|10.4|-73.2|unknown                  |1280    |
|10.3|-73.2|unknown                  |1225    |
|6.4 |-76.4|unknown                  |1103    |
|10.1|-73.7|unknown                  |1073    |
|10.2|-74.0|Guazuma ulmifolia        |1067    |
|10.7|-73.0|Copernicia tectorum      |973     |
|6.5 |-76.3|unknown                  |697     |
|10.1|-73.6|Handroanthus billbergii  |675     |
|10.2|-73.6|Guazuma ulmifolia        |599     |
|11.2|-72.6|Handroanthus billbergii  |571     |
|10.2|-73.